# Algorithms

### Imports

In [62]:
import time
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
from abc import ABC, abstractmethod

np.random.seed(12345)

### Data loading

In [63]:
DistanceMetric = Literal['euclidean', 'manhattan']
def get_dist_function(metric: DistanceMetric):
    if metric == 'euclidean':
        return lambda diff: np.sqrt((diff ** 2).sum(axis=-1))
    elif metric == 'manhattan':
        return lambda diff: np.abs(diff).sum(axis=-1)
    else:
        raise ValueError(f"Invalid metric: {metric}")

#### Capacitated Vehicle Routing Problem (CVRP)

In [64]:
class CvrpInstance:
    """
    Loads a CVRP instance from file.
    Format:
        line 1: vehicle capacity
        line 2 (depot): x, y
        remaining (customers): x, y, demand
    """
    def __init__(self, filepath, metric='euclidean'):
        with open(filepath, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        self.capacity = int(lines[0])
        self.depot = np.array(lines[1].split(), dtype=float)

        # shape (n, 3): columns are x, y, demand
        self.customers = np.loadtxt(lines[2:])

        # coords[0] = depot, coords[1..n] = customers
        self.coords = np.vstack([self.depot, self.customers[:, :2]])
        # demands[0] = 0 (depot), demands[1..n] = customer demands
        self.demands = np.concatenate([[0.0], self.customers[:, 2]])

        self.get_distance = get_dist_function(metric)
        self.dist_matrix = self._compute_distance_matrix()

    def _compute_distance_matrix(self):
        diff = self.coords[:, np.newaxis, :] - self.coords[np.newaxis, :, :]
        return self.get_distance(diff)
    
    def __repr__(self):
        return f"CvrpInstance(capacity={self.capacity}, depot={self.depot}, n={self.customers}, customers={self.customers})"

cvrp_data = CvrpInstance("instances/cvrp.txt")
print(cvrp_data)

CvrpInstance(capacity=160, depot=[30. 40.], n=[[37. 52.  7.]
 [49. 49. 30.]
 [52. 64. 16.]
 [20. 26.  9.]
 [40. 30. 21.]
 [21. 47. 15.]
 [17. 63. 19.]
 [31. 62. 23.]
 [52. 33. 11.]
 [51. 21.  5.]
 [42. 41. 19.]
 [31. 32. 29.]
 [ 5. 25. 23.]
 [12. 42. 21.]
 [36. 16. 10.]
 [52. 41. 15.]
 [27. 23.  3.]
 [17. 33. 41.]
 [13. 13.  9.]
 [57. 58. 28.]
 [62. 42.  8.]
 [42. 57.  8.]
 [16. 57. 16.]
 [ 8. 52. 10.]
 [ 7. 38. 28.]
 [27. 68.  7.]
 [30. 48. 15.]
 [43. 67. 14.]
 [58. 48.  6.]
 [58. 27. 19.]
 [37. 69. 11.]
 [38. 46. 12.]
 [46. 10. 23.]
 [61. 33. 26.]
 [62. 63. 17.]
 [63. 69.  6.]
 [32. 22.  9.]
 [45. 35. 15.]
 [59. 15. 14.]
 [ 5.  6.  7.]
 [10. 17. 27.]
 [21. 10. 13.]
 [ 5. 64. 11.]
 [30. 15. 16.]
 [39. 10. 10.]
 [32. 39.  5.]
 [25. 32. 25.]
 [25. 55. 17.]
 [48. 28. 18.]
 [56. 37. 10.]], customers=[[37. 52.  7.]
 [49. 49. 30.]
 [52. 64. 16.]
 [20. 26.  9.]
 [40. 30. 21.]
 [21. 47. 15.]
 [17. 63. 19.]
 [31. 62. 23.]
 [52. 33. 11.]
 [51. 21.  5.]
 [42. 41. 19.]
 [31. 32. 29.]
 [ 5. 25. 23

#### Vehicle Routing Problem with Time Windows (VRPTW):

In [65]:
class VrptwInstance(CvrpInstance):
    """
    Loads a VRPTW instance from file. Extends CvrpInstance.
    Format:
        line 1: num_vehicles, vehicle capacity
        line 2 (depot): x, y
        remaining (customers): x, y, demand, tw_open, tw_close, service_time
    """
    def __init__(self, filepath, metric='euclidean'):
        with open(filepath, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        header = lines[0].split()
        self.num_vehicles = int(header[0])
        self.capacity = int(header[1])
        self.depot = np.array(lines[1].split(), dtype=float)

        # shape (n, 6): columns are x, y, demand, tw_open, tw_close, service_time
        self.customers = np.loadtxt(lines[2:])

        self.coords = np.vstack([self.depot, self.customers[:, :2]])
        self.demands = np.concatenate([[0.0], self.customers[:, 2]])

        # index 0 = depot (no constraint), 1..n = customers
        self.tw_open = np.concatenate([[0.0], self.customers[:, 3]])
        self.tw_close = np.concatenate([[np.inf], self.customers[:, 4]])
        self.service_time = np.concatenate([[0.0], self.customers[:, 5]])

        self.get_distance = get_dist_function(metric)
        self.dist_matrix = self._compute_distance_matrix()
    
    def __repr__(self):
        return f"VrptwInstance(num_vehicles={self.num_vehicles}, capacity={self.capacity}, depot={self.depot}, n={self.customers.shape[0]}, customers={self.customers})"

vrptw_data = VrptwInstance("instances/vrptw.txt")
print(vrptw_data)

VrptwInstance(num_vehicles=25, capacity=200, depot=[35. 35.], n=100, customers=[[ 41.  49.  10. 161. 171.  10.]
 [ 35.  17.   7.  50.  60.  10.]
 [ 55.  45.  13. 116. 126.  10.]
 [ 55.  20.  19. 149. 159.  10.]
 [ 15.  30.  26.  34.  44.  10.]
 [ 25.  30.   3.  99. 109.  10.]
 [ 20.  50.   5.  81.  91.  10.]
 [ 10.  43.   9.  95. 105.  10.]
 [ 55.  60.  16.  97. 107.  10.]
 [ 30.  60.  16. 124. 134.  10.]
 [ 20.  65.  12.  67.  77.  10.]
 [ 50.  35.  19.  63.  73.  10.]
 [ 30.  25.  23. 159. 169.  10.]
 [ 15.  10.  20.  32.  42.  10.]
 [ 30.   5.   8.  61.  71.  10.]
 [ 10.  20.  19.  75.  85.  10.]
 [  5.  30.   2. 157. 167.  10.]
 [ 20.  40.  12.  87.  97.  10.]
 [ 15.  60.  17.  76.  86.  10.]
 [ 45.  65.   9. 126. 136.  10.]
 [ 45.  20.  11.  62.  72.  10.]
 [ 45.  10.  18.  97. 107.  10.]
 [ 55.   5.  29.  68.  78.  10.]
 [ 65.  35.   3. 153. 163.  10.]
 [ 65.  20.   6. 172. 182.  10.]
 [ 45.  30.  17. 132. 142.  10.]
 [ 35.  40.  16.  37.  47.  10.]
 [ 41.  37.  16.  39.  49.  10

#### Stoppage criterions

In [66]:
class StoppingCriterion(ABC):
    def reset(self):
        pass
    @abstractmethod
    def should_stop(self, gen: int, history: list[float]) -> bool: ...
    @abstractmethod
    def __repr__(self) -> str: ...


class MaxGenerations(StoppingCriterion):
    def __init__(self, n: int):
        assert n > 0, "n must be > 0"
        self.n = n
    def should_stop(self, gen, history):
        return gen + 1 >= self.n
    def __repr__(self):
        return f"MaxGenerations({self.n})"


class TimeLimit(StoppingCriterion):
    def __init__(self, seconds: float):
        assert seconds > 0, "seconds must be > 0"
        self.seconds = seconds
        self._start = None
    def reset(self):
        self._start = time.monotonic()
    def should_stop(self, gen, history):
        return (time.monotonic() - self._start) >= self.seconds
    def __repr__(self):
        return f"TimeLimit({self.seconds}s)"


class MinImprovement(StoppingCriterion):
    def __init__(self, window: int, min_pct: float = 0.05):
        assert window > 0, "window must be > 0"
        assert 0 <= min_pct < 1, "min_pct must be in [0, 1)"
        self.window = window
        self.min_pct = min_pct
    def reset(self):
        pass
    def should_stop(self, gen, history):
        if len(history) < self.window:
            return False
        oldest = history[-self.window]
        current = history[-1]
        if oldest == 0:
            return False
        improvement = (oldest - current) / oldest
        return improvement <= self.min_pct
    def __repr__(self):
        return f"MinImprovement(window={self.window}, min_pct={self.min_pct})"


class TargetObjective(StoppingCriterion):
    def __init__(self, target: float):
        assert target > 0, "target must be > 0"
        self.target = target
    def should_stop(self, gen, history):
        return len(history) > 0 and history[-1] <= self.target
    def __repr__(self):
        return f"TargetObjective({self.target})"

### Fitness / cost evaluation

In [67]:
Route = np.ndarray
DistanceMatrix = np.ndarray
Solution = list[Route]

def route_cost(route: Route, dist_matrix: DistanceMatrix) -> float:
    """Total distance of a single route (depot -> customers -> depot). Depot is node 0."""
    if route.shape[0] == 0:
        return 0.0
    nodes = np.concatenate(([0], route, [0]))
    return dist_matrix[nodes[:-1], nodes[1:]].sum()

def solution_cost(routes: Solution, dist_matrix: DistanceMatrix) -> float:
    """Total distance across all routes."""
    return sum(route_cost(r, dist_matrix) for r in routes)

def is_feasible_cvrp(routes: Solution, instance: CvrpInstance) -> bool:
    """Check capacity constraint for each route."""
    for route in routes:
        if instance.demands[route].sum() > instance.capacity:
            return False
    return True

def is_feasible_vrptw(routes: Solution, instance: VrptwInstance) -> bool:
    """Check capacity + time window constraints for each route."""
    for route in routes:
        if instance.demands[route].sum() > instance.capacity:
            return False
        route_time = 0.0
        prev = 0
        for node in route:
            route_time += instance.dist_matrix[prev, node]
            if route_time > instance.tw_close[node]:
                return False
            route_time = max(route_time, instance.tw_open[node]) + instance.service_time[node]
            prev = node
    return True

### Local search operators

In [68]:
def two_opt(route: Route, dist_matrix: DistanceMatrix) -> Route:
    """Intra-route 2-opt: reverse a segment to reduce cost. First-improvement, repeat until stable."""
    if route.shape[0] < 2:
        return route.copy()
    best = route.copy()
    improved = True
    while improved:
        improved = False
        n = best.shape[0]
        for i in range(n - 1):
            for j in range(i + 1, n):
                # edges being removed: (prev_i -> best[i]) and (best[j] -> next_j)
                prev_i = 0 if i == 0 else best[i - 1]
                next_j = 0 if j == n - 1 else best[j + 1]
                old = dist_matrix[prev_i, best[i]] + dist_matrix[best[j], next_j]
                new = dist_matrix[prev_i, best[j]] + dist_matrix[best[i], next_j]
                if new < old - 1e-10:
                    best[i:j+1] = best[i:j+1][::-1]
                    improved = True
    return best

def or_opt(route: Route, dist_matrix: DistanceMatrix) -> Route:
    """Intra-route Or-opt: relocate subsequences of length 1, 2, or 3."""
    if route.shape[0] < 2:
        return route.copy()
    best = route.copy()
    best_cost = route_cost(best, dist_matrix)
    improved = True
    while improved:
        improved = False
        n = best.shape[0]
        for seg_len in [1, 2, 3]:
            for i in range(n - seg_len + 1):
                segment = best[i:i+seg_len].copy()
                remaining = np.concatenate([best[:i], best[i+seg_len:]])
                for j in range(remaining.shape[0] + 1):
                    candidate = np.concatenate([remaining[:j], segment, remaining[j:]])
                    c = route_cost(candidate, dist_matrix)
                    if c < best_cost - 1e-10:
                        best = candidate
                        best_cost = c
                        improved = True
                        break
                if improved:
                    break
            if improved:
                break
    return best

def relocate(routes: Solution, instance: CvrpInstance) -> Solution:
    """Inter-route: move a single customer from one route to another if it reduces total cost."""
    dm = instance.dist_matrix
    best = [r.copy() for r in routes]
    while True:
        improved = False
        for ri in range(len(best)):
            if best[ri].shape[0] == 0:
                continue
            for ci in range(best[ri].shape[0]):
                node = best[ri][ci]
                src_without = np.concatenate([best[ri][:ci], best[ri][ci+1:]])
                remove_saving = route_cost(best[ri], dm) - route_cost(src_without, dm)
                for rj in range(len(best)):
                    if ri == rj:
                        continue
                    # check capacity constraint
                    if instance.demands[best[rj]].sum() + instance.demands[node] > instance.capacity:
                        continue
                    # find best insertion position
                    for pos in range(best[rj].shape[0] + 1):
                        dst_with = np.concatenate([best[rj][:pos], [node], best[rj][pos:]])
                        insert_cost = route_cost(dst_with, dm) - route_cost(best[rj], dm)
                        if insert_cost < remove_saving - 1e-10:
                            best[ri] = src_without
                            best[rj] = dst_with
                            improved = True
                            break
                    if improved:
                        break
                if improved:
                    break
            if improved:
                break
        if not improved:
            break
    return [r for r in best if r.shape[0] > 0]

def local_search(routes: Solution, instance: CvrpInstance) -> Solution:
    """Apply local search operators: 2-opt and or-opt per route, then inter-route relocate."""
    dm = instance.dist_matrix
    improved = [or_opt(two_opt(r, dm), dm) for r in routes]
    return relocate(improved, instance)

### Base solver class

In [69]:
DEFAULT_CRITERIA = [TimeLimit(120), MinImprovement(50, min_pct=0.05)]

class SolverBase(ABC):
    """Base class for VRP metaheuristic solvers."""

    def __init__(self, criteria: list[StoppingCriterion] | None = None):
        self.criteria = criteria if criteria else DEFAULT_CRITERIA

    @abstractmethod
    def _construct(self, instance: CvrpInstance | VrptwInstance) -> Solution:
        """Construct a new solution."""
        ...

    @abstractmethod
    def _improve(self, routes: Solution, instance: CvrpInstance | VrptwInstance) -> Solution:
        """Improve a solution (e.g. local search)."""
        ...

    def solve(self, instance: CvrpInstance | VrptwInstance) -> tuple[Solution, float, list[float]]:
        """Main loop: construct, improve, track best. Stops when any criterion is met."""
        self.is_vrptw = isinstance(instance, VrptwInstance)
        best_routes: Solution = []
        best_cost: float = np.inf
        history: list[float] = []

        for c in self.criteria:
            c.reset()

        gen = 0
        while True:
            routes = self._construct(instance)
            routes = self._improve(routes, instance)

            cost = solution_cost(routes, instance.dist_matrix)
            if cost < best_cost:
                best_cost = cost
                best_routes = [r.copy() for r in routes]
            history.append(best_cost)

            if any(c.should_stop(gen, history) for c in self.criteria):
                break
            gen += 1

        return best_routes, best_cost, history

### Greedy randomized adaptive search procedure (GRASP)

In [70]:
class GraspSolver(SolverBase):
    """
    GRASP metaheuristic for VRP.
    Phase 1: Greedy Randomized Adaptive construction
    Phase 2: Local search improvement
    """

    def __init__(self, alpha: float = 0.3, criteria: list[StoppingCriterion] | None = None):
        super().__init__(criteria)
        self.alpha = alpha

    def _construct(self, instance: CvrpInstance | VrptwInstance) -> Solution:
        """
        Greedy Randomized Adaptive construction.
        - Greedy function: distance from current node to candidates (lower = better)
        - RCL: candidates with cost <= c_min + alpha * (c_max - c_min)
        - Probabilistic selection: pick uniformly at random from RCL
        - Adaptive: greedy values re-evaluated after each insertion
        """
        dm = instance.dist_matrix
        n = len(instance.demands) - 1

        unvisited = set(range(1, n + 1))
        routes: Solution = []

        while unvisited:
            route = []
            load = 0.0
            current = 0  # depot
            time = 0.0

            while True:
                # feasible candidates
                candidates = []
                for node in unvisited:
                    if load + instance.demands[node] > instance.capacity:
                        continue
                    if self.is_vrptw:
                        arrival = time + dm[current, node]
                        if arrival > instance.tw_close[node]:
                            continue
                    candidates.append(node)

                if len(candidates) == 0:
                    break

                # greedy function: distance from current position
                costs = dm[current, candidates]
                c_min, c_max = costs.min(), costs.max()

                # restricted candidate list
                threshold = c_min + self.alpha * (c_max - c_min)
                rcl = np.array(candidates)[costs <= threshold + 1e-10]

                # probabilistic selection
                chosen = np.random.choice(rcl)

                route.append(chosen)
                load += instance.demands[chosen]
                if self.is_vrptw:
                    arrival = time + dm[current, chosen]
                    time = max(arrival, instance.tw_open[chosen]) + instance.service_time[chosen]
                current = chosen
                unvisited.remove(chosen)

            if len(route) > 0:
                routes.append(np.array(route, dtype=int))

        return routes

    def _improve(self, routes: Solution, instance: CvrpInstance | VrptwInstance) -> Solution:
        """Improve solution via local search."""
        return local_search(routes, instance)

In [71]:
grasp = GraspSolver(alpha=0.3, criteria=[MaxGenerations(50)])

print("=== CVRP ===")
cvrp_routes, cvrp_cost, cvrp_history = grasp.solve(cvrp_data)
print(f"Cost: {cvrp_cost:.2f}, Routes: {len(cvrp_routes)}, Feasible: {is_feasible_cvrp(cvrp_routes, cvrp_data)}")

print("\n=== VRPTW ===")
grasp_vrptw = GraspSolver(alpha=0.3, criteria=[MaxGenerations(50)])
vrptw_routes, vrptw_cost, vrptw_history = grasp_vrptw.solve(vrptw_data)
print(f"Cost: {vrptw_cost:.2f}, Routes: {len(vrptw_routes)}, Feasible: {is_feasible_vrptw(vrptw_routes, vrptw_data)}")

=== CVRP ===
Cost: 648.55, Routes: 5, Feasible: True

=== VRPTW ===
Cost: 925.40, Routes: 9, Feasible: False


### Ant Colony Optimization (ACO)

In [72]:
class AcoSolver(SolverBase):
    """
    Ant Colony Optimization for VRP.
    Works for both CVRP and VRPTW.
    """

    def __init__(self, n_ants: int = 20, alpha: float = 1.0, beta: float = 2.0, rho: float = 0.1,
                 criteria: list[StoppingCriterion] | None = None):
        super().__init__(criteria)
        self.n_ants = n_ants
        self.alpha = alpha  # pheromone importance
        self.beta = beta    # heuristic importance
        self.rho = rho      # evaporation rate

    def _construct(self, instance: CvrpInstance | VrptwInstance) -> Solution:
        """Construct solution using ant's probabilistic rule (pheromone * heuristic)."""
        # TODO
        pass

    def _improve(self, routes: Solution, instance: CvrpInstance | VrptwInstance) -> Solution:
        """Improve solution via local search."""
        # TODO
        return local_search(routes, instance)

    def _update_pheromones(self, solutions: list[Solution], costs: list[float]) -> None:
        """Evaporate and deposit pheromones."""
        # TODO
        pass

    def solve(self, instance: CvrpInstance | VrptwInstance) -> tuple[Solution, float, list[float]]:
        """ACO main loop: all ants construct, update pheromones, track best."""
        # TODO: override base solve() to handle multiple ants per iteration + pheromone updates
        pass

# Visualizations

# Results